# Electricity Consumption – Data Preprocessing

## 1. Imports and Dataset

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("../dataset/raw/household_power_consumption.txt")
if not DATA_PATH.exists() and Path("/mnt/data/household_power_consumption.txt").exists():
    DATA_PATH = Path("/mnt/data/household_power_consumption.txt")
OUTPUT_PATH = Path("electricity_preprocessed_hourly.csv")
if str(Path.cwd()) == "/mnt/data":
    OUTPUT_PATH = Path("/mnt/data/electricity_preprocessed_hourly.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError("Place household_power_consumption.txt in the same folder as this notebook.")

numeric_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

## 2. Load and Inspect Raw Data

In [5]:
# Read in chunks to avoid loading the 133 MB file into memory at once.
chunk_size = 200_000
sum_parts = []
count_parts = []
raw_rows = 0
raw_missing = pd.Series(0, index=numeric_cols, dtype="int64")

for chunk in pd.read_csv(
    DATA_PATH,
    sep=";",
    na_values=["?"],
    low_memory=False,
    chunksize=chunk_size
):
    raw_rows += len(chunk)
    raw_missing += chunk[numeric_cols].isna().sum()

    # Remove duplicate timestamps within each chunk.
    chunk = chunk.drop_duplicates(subset=["Date", "Time"])

    dt = pd.to_datetime(
        chunk["Date"].astype(str) + " " + chunk["Time"].astype(str),
        dayfirst=True,
        errors="coerce"
    )
    valid = dt.notna()
    chunk = chunk.loc[valid, numeric_cols].copy()
    dt = dt.loc[valid]

    chunk.index = dt.dt.floor("h")
    sum_parts.append(chunk.groupby(level=0).sum(min_count=1))
    count_parts.append(chunk.groupby(level=0).count())

print("Raw rows:", raw_rows)
print("\\nMissing values by column:")
print(raw_missing)
print("\\nTotal missing numeric cells:", int(raw_missing.sum()))

Raw rows: 2075259
\nMissing values by column:
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64
\nTotal missing numeric cells: 181853


## 3. Hourly Aggregation and Missing-Value Handling

In [6]:
# Combine hourly aggregates from all chunks.
hourly_sums = pd.concat(sum_parts).groupby(level=0).sum(min_count=1).sort_index()
hourly_counts = pd.concat(count_parts).groupby(level=0).sum().sort_index()

full_index = pd.date_range(
    hourly_sums.index.min(),
    hourly_sums.index.max(),
    freq="h"
)
hourly = pd.DataFrame(index=full_index)
hourly.index.name = "datetime"

active_count = hourly_counts["Global_active_power"].reindex(full_index, fill_value=0)
active_sum = hourly_sums["Global_active_power"].reindex(full_index)

# Require at least 55 valid minute readings per hour.
# Mean kW over one hour is numerically equal to hourly kWh.
hourly["consumption_kwh"] = active_sum / active_count.replace(0, np.nan)
hourly.loc[active_count < 55, "consumption_kwh"] = np.nan

# Hourly mean electrical measurements.
for col in ["Global_reactive_power", "Voltage", "Global_intensity"]:
    col_sum = hourly_sums[col].reindex(full_index)
    col_count = hourly_counts[col].reindex(full_index)
    hourly[col] = col_sum / col_count.replace(0, np.nan)
    hourly.loc[col_count < 55, col] = np.nan

# Sub-metering readings are Wh per minute.
# Average valid minute reading * 60 minutes / 1000 converts to hourly kWh.
for col in ["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]:
    col_sum = hourly_sums[col].reindex(full_index)
    col_count = hourly_counts[col].reindex(full_index)
    hourly[col + "_kwh"] = (
        col_sum / col_count.replace(0, np.nan) * 60 / 1000
    )
    hourly.loc[col_count < 55, col + "_kwh"] = np.nan

# Fill only short gaps in predictor measurements.
# Do NOT fill the target consumption_kwh column.
predictor_cols = [
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1_kwh",
    "Sub_metering_2_kwh",
    "Sub_metering_3_kwh"
]
hourly[predictor_cols] = hourly[predictor_cols].interpolate(
    method="time",
    limit=2,
    limit_area="inside"
)

print(hourly.head())
print("\\nHourly rows:", len(hourly))
print("Missing target hours:", int(hourly["consumption_kwh"].isna().sum()))

                     consumption_kwh  Global_reactive_power     Voltage  \
datetime                                                                  
2006-12-16 17:00:00              NaN                    NaN         NaN   
2006-12-16 18:00:00         3.632200               0.080033  234.580167   
2006-12-16 19:00:00         3.400233               0.085233  233.232500   
2006-12-16 20:00:00         3.268567               0.075100  234.071500   
2006-12-16 21:00:00         3.056467               0.076667  237.158667   

                     Global_intensity  Sub_metering_1_kwh  Sub_metering_2_kwh  \
datetime                                                                        
2006-12-16 17:00:00               NaN                 NaN                 NaN   
2006-12-16 18:00:00         15.600000                 0.0               0.403   
2006-12-16 19:00:00         14.503333                 0.0               0.086   
2006-12-16 20:00:00         13.916667                 0.0            

## 4. Feature Engineering

In [7]:
features = pd.DataFrame(index=hourly.index)

# Calendar and time features
features["hour"] = features.index.hour
features["day_of_week"] = features.index.dayofweek
features["day_of_month"] = features.index.day
features["month"] = features.index.month
features["is_weekend"] = (features["day_of_week"] >= 5).astype("int8")

# Cyclical encoding: keeps 23:00 close to 00:00, and December close to January.
features["hour_sin"] = np.sin(2 * np.pi * features["hour"] / 24)
features["hour_cos"] = np.cos(2 * np.pi * features["hour"] / 24)
features["dow_sin"] = np.sin(2 * np.pi * features["day_of_week"] / 7)
features["dow_cos"] = np.cos(2 * np.pi * features["day_of_week"] / 7)
features["month_sin"] = np.sin(2 * np.pi * (features["month"] - 1) / 12)
features["month_cos"] = np.cos(2 * np.pi * (features["month"] - 1) / 12)

consumption = hourly["consumption_kwh"]

# Historical consumption (all values come from earlier hours).
for lag in [1, 2, 3, 24, 48, 168]:
    features[f"consumption_lag_{lag}"] = consumption.shift(lag)

# Rolling statistics use shift(1) to avoid including the target hour.
previous = consumption.shift(1)
features["rolling_mean_6"] = previous.rolling(6, min_periods=6).mean()
features["rolling_mean_24"] = previous.rolling(24, min_periods=24).mean()
features["rolling_mean_168"] = previous.rolling(168, min_periods=168).mean()
features["rolling_std_24"] = previous.rolling(24, min_periods=24).std()
features["rolling_max_24"] = previous.rolling(24, min_periods=24).max()

# Previous-hour electrical measurements.
for col in predictor_cols:
    features[f"{col}_lag_1"] = hourly[col].shift(1)

# Target: actual electricity energy used during the hour represented by the row.
features["target_consumption_kwh"] = consumption

# Remove incomplete rows; target values are never imputed.
features = features.replace([np.inf, -np.inf], np.nan)
processed = features.dropna().copy()
processed.index.name = "datetime"

print(processed.head())
print("\\nProcessed shape:", processed.shape)
print("Remaining missing values:", int(processed.isna().sum().sum()))

                     hour  day_of_week  day_of_month  month  is_weekend  \
datetime                                                                  
2006-12-23 18:00:00    18            5            23     12           1   
2006-12-23 19:00:00    19            5            23     12           1   
2006-12-23 20:00:00    20            5            23     12           1   
2006-12-23 21:00:00    21            5            23     12           1   
2006-12-23 22:00:00    22            5            23     12           1   

                     hour_sin      hour_cos   dow_sin   dow_cos  month_sin  \
datetime                                                                     
2006-12-23 18:00:00 -1.000000 -1.836970e-16 -0.974928 -0.222521       -0.5   
2006-12-23 19:00:00 -0.965926  2.588190e-01 -0.974928 -0.222521       -0.5   
2006-12-23 20:00:00 -0.866025  5.000000e-01 -0.974928 -0.222521       -0.5   
2006-12-23 21:00:00 -0.707107  7.071068e-01 -0.974928 -0.222521       -0.5   
2006-1

C:\Users\sadiq\AppData\Local\Temp\ipykernel_25632\617215383.py:5: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  features["day_of_week"] = features.index.dayofweek


## 5. Final Dataset Validation

In [8]:
target_col = "target_consumption_kwh"
feature_cols = [c for c in processed.columns if c != target_col]

print("Rows:", len(processed))
print("Feature count:", len(feature_cols))
print("Target:", target_col)
print("Date range:", processed.index.min(), "to", processed.index.max())
print("Missing cells:", int(processed.isna().sum().sum()))
print("\\nFeature columns:")
print(feature_cols)

assert processed.isna().sum().sum() == 0
assert target_col in processed.columns
assert processed.index.is_monotonic_increasing

Rows: 31454
Feature count: 28
Target: target_consumption_kwh
Date range: 2006-12-23 18:00:00 to 2010-11-26 20:00:00
Missing cells: 0
\nFeature columns:
['hour', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'consumption_lag_1', 'consumption_lag_2', 'consumption_lag_3', 'consumption_lag_24', 'consumption_lag_48', 'consumption_lag_168', 'rolling_mean_6', 'rolling_mean_24', 'rolling_mean_168', 'rolling_std_24', 'rolling_max_24', 'Global_reactive_power_lag_1', 'Voltage_lag_1', 'Global_intensity_lag_1', 'Sub_metering_1_kwh_lag_1', 'Sub_metering_2_kwh_lag_1', 'Sub_metering_3_kwh_lag_1']


## 6. Save Final Preprocessed Dataset

In [9]:
processed.to_csv(OUTPUT_PATH)
print("Saved:", OUTPUT_PATH.resolve())

Saved: C:\Users\sadiq\OneDrive\Attachments\Desktop\ml projecr\preporcessing\electricity_preprocessed_hourly.csv
